# 04 — TTS: all four output languages (VI, EN, ZH, KO)

Tests the TTS stage. Because the device is **bidirectional** half-duplex
PTT (VI↔EN, VI↔ZH, VI↔KO), it has to *speak* every target language, not
just Vietnamese — so this notebook validates a synthesized voice for
**VI, EN, ZH (Mandarin), and KO (Korean)**.

Stack decision (see the section-5 note below for why):
- **VI, EN, ZH → Piper** (VITS/ONNX, natively distributed as ONNX — no
  export needed, just pick the voice). Piper has good Mandarin (`zh_CN`)
  but **no Korean voice at all**.
- **KO → Meta MMS-TTS** (`facebook/mms-tts-kor`, also VITS) — same
  architecture family, fills the one gap Piper leaves. Needs a one-time
  ONNX export (done in section 5) and `uroman` romanization at the text
  front-end (Korean is `is_uroman=True`).

**Read `docs/decisions/piper-vi-voice-license-trap.md` before using
this for anything beyond a prototype.** Two license issues found while
picking a Vietnamese voice, neither visible from the repo-level `license:
mit` tag on `rhasspy/piper-voices`:

1. Of the three Vietnamese voices available, only `vi_VN-vais1000-medium`
   is commercially usable (CC BY 4.0). `vivos-x_low` is CC-BY-NC-SA
   (NonCommercial — blocked) and `25hours_single-low` has no declared
   license at all. `vais1000-medium` is used below, and happens to also
   be the best quality tier available for Vietnamese in this repo.
2. Piper's phonemizer (`piper-phonemize`, used here via the official
   `piper-tts` pip package) links `espeak-ng`, which is GPL-3.0. **For
   this hackathon that is not a build blocker** — GPL obligations bind
   at commercial distribution, not at a prototype/demo; revisit at
   commercialization. The decision doc tracks it for that later stage.
   (MMS-TTS for Korean is MIT-licensed and does not use espeak.)
</cell id="d89ce2d1">

In [ ]:
from pathlib import Path

VOICE_REPO = "rhasspy/piper-voices"
VOICE_PATH = "vi/vi_VN/vais1000/medium/vi_VN-vais1000-medium"
FINAL_DIR = Path("../models/tts/vi")


## 1. Download the pre-made ONNX voice -- no conversion

In [ ]:
from huggingface_hub import hf_hub_download

onnx_path = hf_hub_download(VOICE_REPO, f"{VOICE_PATH}.onnx")
config_path = hf_hub_download(VOICE_REPO, f"{VOICE_PATH}.onnx.json")
model_card_path = hf_hub_download(VOICE_REPO, "vi/vi_VN/vais1000/medium/MODEL_CARD")

print(Path(model_card_path).read_text())


## 2. Synthesize test sentences

Same four VI sentences used throughout `experiments/03_mt.ipynb` and
`experiments/02_asr.ipynb` (1 general + 3 industrial-glossary-flavored),
for continuity across the pipeline stages.


In [ ]:
import wave

from piper import PiperVoice

voice = PiperVoice.load(onnx_path, config_path=config_path)

test_sentences = [
    "Xin chào, bạn khỏe không?",
    "Vui lòng đeo kính bảo hộ trước khi vận hành máy.",
    "Xe nâng đang đến gần khu vực số ba.",
    "Dừng máy ngay lập tức nếu nghe thấy báo động.",
]

OUT_DIR = Path("tts_wav_out")
OUT_DIR.mkdir(exist_ok=True)

wav_paths = []
for i, sent in enumerate(test_sentences):
    path = OUT_DIR / f"{i}.wav"
    with wave.open(str(path), "wb") as wav_file:
        voice.synthesize_wav(sent, wav_file)
    wav_paths.append(path)
    print(f"{sent} -> {path} ({path.stat().st_size} bytes)")


## 3. Round-trip intelligibility check (TTS -> ASR)

No human listener available in this environment, so validate
intelligibility indirectly: feed the synthesized audio back through the
already-validated PhoWhisper ASR from `experiments/02_asr.ipynb` and
compare against the original text. This stacks two error sources (TTS
pronunciation quality *and* ASR recognition accuracy), so it's a lower
bound on TTS quality, not a precise measurement -- but a real listener
check should still happen before trusting this voice for the demo.


In [ ]:
import librosa
import numpy as np
import soundfile as sf
from huggingface_hub import snapshot_download
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
from transformers import WhisperProcessor

asr_dir = snapshot_download(
    "huuquyet/PhoWhisper-base",
    allow_patterns=[
        "onnx/encoder_model.onnx", "onnx/decoder_model_merged.onnx",
        "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
        "config.json", "generation_config.json", "preprocessor_config.json",
        "vocab.json", "merges.txt", "normalizer.json", "added_tokens.json",
    ],
)
asr_processor = WhisperProcessor.from_pretrained(asr_dir)
asr_model = ORTModelForSpeechSeq2Seq.from_pretrained(asr_dir, use_cache=True, use_merged=True, provider="CPUExecutionProvider")

for sent, path in zip(test_sentences, wav_paths):
    audio, sr = sf.read(path)
    if sr != 16000:
        audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=16000)
        sr = 16000
    inputs = asr_processor(audio, sampling_rate=sr, return_tensors="pt")
    gen = asr_model.generate(**inputs, language="vi", task="transcribe", max_new_tokens=64)
    heard = asr_processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
    print(f"original: {sent}\ntts->asr: {heard}\n")


The general greeting and two of the three industrial sentences come
back essentially verbatim (minor diacritic drift: "đeo" heard as "đéo",
"đang" heard as "đăng" -- likely ASR mishearing correctly-pronounced
but less-common words rather than a TTS mispronunciation). The fourth
sentence ("Dừng máy..." / "Stop the machine...") comes back
significantly wrong ("Xuân máy..."). Same pattern as the MT baseline:
generic/general-purpose models are noticeably weaker on
industrial/safety vocabulary specifically -- another data point for why
the glossary engine differentiator matters, not just for MT.


## 4. Package into `models/tts/vi/`

In [ ]:
import shutil
from datetime import date

FINAL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copyfile(onnx_path, FINAL_DIR / "vi_VN-vais1000-medium.onnx")
shutil.copyfile(config_path, FINAL_DIR / "vi_VN-vais1000-medium.onnx.json")

manifest = f"""# tts/vi

- Source: {VOICE_REPO}, {VOICE_PATH} (Piper voices are natively ONNX --
  not exported by us)
- Packaged: {date.today().isoformat()}
- License: CC BY 4.0 (VAIS-1000 dataset) -- the *only* commercially
  usable Vietnamese voice in this repo. Do NOT substitute
  `vivos-x_low` (CC-BY-NC-SA, blocked) or `25hours_single-low`
  (unlicensed). See docs/decisions/piper-vi-voice-license-trap.md.
- **Open licensing question, not resolved here:** Piper's phonemizer
  (piper-phonemize/espeak-ng) is GPL-3.0. This voice's config declares
  `"espeak": {{"voice": "vi"}}` -- Vietnamese phonemization requires it,
  there's no way around it for this specific model. Whether embedding
  it in the shipped on-device binary GPL-taints the product is an open
  question -- needs a real answer before Phase 3 commits to this
  architecture. Not a blocker for prototyping.
- Quality: medium (best available for VI in this repo), single speaker,
  22050Hz
- Eval: round-trip TTS -> PhoWhisper-ASR intelligibility check on 4 VI
  sentences (1 general + 3 industrial) -- 3/4 essentially verbatim, 1
  industrial sentence significantly misheard. See
  experiments/04_tts.ipynb section 3. No human-listener check done yet.

Files:
- `vi_VN-vais1000-medium.onnx`, `vi_VN-vais1000-medium.onnx.json`
  (phoneme_id_map + audio/inference config -- both needed, this isn't
  a pure ONNX-in-ONNX-out interface, the JSON side carries the
  phoneme vocabulary)
"""
(FINAL_DIR / "MANIFEST.md").write_text(manifest)

shutil.rmtree(OUT_DIR)
sorted(p.name for p in FINAL_DIR.iterdir())


## 5. The other output directions: EN, ZH, KO

The device translates *both ways* on every pair, so it has to speak
English, Mandarin, and Korean too — not just Vietnamese. Coverage:

- **EN, ZH → Piper**, same as VI: pick a pre-made ONNX voice, no export.
  `en_US-lessac-medium` and `zh_CN-huayan-medium` (Mandarin). Piper
  reaches these through the same espeak-ng front-end as Vietnamese.
- **KO → Piper has no Korean voice at all** ([standing open request](https://github.com/rhasspy/piper/discussions/680)),
  so Korean uses **`facebook/mms-tts-kor`** — a Meta MMS VITS model
  (MIT). Same architecture family as Piper (VITS), but two differences
  that matter for the Rust port: (1) it ships as a PyTorch checkpoint,
  so we export it to ONNX here (like the MT stage); (2) its tokenizer is
  `is_uroman=True` — Korean text must be romanized with `uroman` before
  tokenizing (its own vocab is only ~25 Latin symbols), so `uroman` is a
  required text-front-end dependency for KO, in place of espeak.

Each language packages into `models/tts/<lang>/`, mirroring `tts/vi/`.


In [ ]:
# --- EN and ZH via Piper (pre-made ONNX, same path as VI) ---
en_zh_voices = {
    "en": "en/en_US/lessac/medium/en_US-lessac-medium",
    "zh": "zh/zh_CN/huayan/medium/zh_CN-huayan-medium",
}
# one general + one industrial-safety sentence per language, for the
# round-trip check in the next cell
tts_probe = {
    "en": ["Hello, how are you?", "Stop the machine immediately if you hear an alarm."],
    "zh": ["你好，你好吗？", "如果听到警报，请立即停止机器。"],
    "ko": ["안녕하세요, 잘 지내세요?", "경보가 울리면 즉시 기계를 멈추세요."],
}

piper_out = Path("tts_wav_out2")
piper_out.mkdir(exist_ok=True)
piper_synth = {}  # lang -> list[Path]

for lang, voice_path in en_zh_voices.items():
    onnx_p = hf_hub_download(VOICE_REPO, f"{voice_path}.onnx")
    cfg_p = hf_hub_download(VOICE_REPO, f"{voice_path}.onnx.json")
    v = PiperVoice.load(onnx_p, config_path=cfg_p)

    paths = []
    for i, sent in enumerate(tts_probe[lang]):
        wp = piper_out / f"{lang}_{i}.wav"
        with wave.open(str(wp), "wb") as wf:
            v.synthesize_wav(sent, wf)
        paths.append(wp)
        print(f"[{lang}] {sent} -> {wp} ({wp.stat().st_size} bytes)")
    piper_synth[lang] = paths

    # package
    final = Path(f"../models/tts/{lang}")
    final.mkdir(parents=True, exist_ok=True)
    base = Path(voice_path).name
    shutil.copyfile(onnx_p, final / f"{base}.onnx")
    shutil.copyfile(cfg_p, final / f"{base}.onnx.json")
    (final / "MANIFEST.md").write_text(
        f"""# tts/{lang}

- Source: {VOICE_REPO}, {voice_path} (Piper VITS voice, natively ONNX --
  not exported by us)
- Packaged: {date.today().isoformat()}
- License: MIT (rhasspy/piper-voices repo tag); underlying voice's own
  dataset license not separately audited -- fine for the hackathon
  prototype, audit before commercialization (same espeak-ng GPL note as
  tts/vi applies to the phonemizer, not the model).
- Phonemizer: espeak-ng ({'en' if lang == 'en' else 'zh' if lang == 'zh' else lang} voice), same front-end as tts/vi.
- Quality: medium tier, single speaker, 22050Hz.
- Eval: round-trip TTS -> Whisper-small ASR intelligibility check, see
  experiments/04_tts.ipynb section 5.

Files: `{base}.onnx`, `{base}.onnx.json`.
"""
    )
    print(f"  packaged -> models/tts/{lang}/\n")


In [ ]:
# --- KO via MMS-TTS (facebook/mms-tts-kor), exported to ONNX ---
# uroman is required: the tokenizer is is_uroman=True (Korean text must
# be romanized before tokenizing). transformers applies it automatically
# when the `uroman` package is installed (pip install uroman).
import onnxruntime as ort
import torch
from transformers import VitsModel, AutoTokenizer

KO_REPO = "facebook/mms-tts-kor"
ko_tok = AutoTokenizer.from_pretrained(KO_REPO)
assert getattr(ko_tok, "is_uroman", False), "expected MMS Korean tokenizer to require uroman"
ko_model = VitsModel.from_pretrained(KO_REPO).eval()
# Deterministic synthesis (matches the ONNX export below) -- VITS is
# stochastic by default (flow + duration noise); zeroing both makes
# PyTorch and ONNX agree and makes the demo reproducible.
ko_model.noise_scale = 0.0
ko_model.noise_scale_duration = 0.0
ko_sr = ko_model.config.sampling_rate

ko_final = Path("../models/tts/ko")
ko_final.mkdir(parents=True, exist_ok=True)
ko_onnx_path = ko_final / "mms_tts_kor.onnx"

# Export once, with a dynamic sequence axis so any-length input works
# (verified against a second, different-length sentence below -- the
# same fixed-length trap that bit the Whisper encoder).
example_ids = ko_tok(tts_probe["ko"][0], return_tensors="pt")["input_ids"]
torch.onnx.export(
    ko_model, (example_ids,), str(ko_onnx_path), dynamo=False,
    input_names=["input_ids"], output_names=["waveform"],
    dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "waveform": {0: "batch", 1: "samples"}},
    opset_version=15,
)

ko_sess = ort.InferenceSession(str(ko_onnx_path), providers=["CPUExecutionProvider"])
ko_synth = []
for i, sent in enumerate(tts_probe["ko"]):
    ids = ko_tok(sent, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        ref = ko_model(input_ids=ids).waveform.squeeze().cpu().numpy()
    got = ko_sess.run(None, {"input_ids": ids.numpy()})[0].squeeze()
    n = min(len(ref), len(got))
    diff = float(np.max(np.abs(ref[:n] - got[:n])))
    wp = piper_out / f"ko_{i}.wav"
    sf.write(str(wp), got, ko_sr)
    ko_synth.append(wp)
    print(f"[ko] {sent} -> {wp} ({len(got)} samples @ {ko_sr}Hz, onnx-vs-torch maxdiff {diff:.2e})")
    assert diff < 1e-2, "ONNX export diverged from PyTorch reference"

# Save the tokenizer (vocab + is_uroman/normalize config) alongside the
# ONNX so the Rust port has everything the text front-end needs.
ko_tok.save_pretrained(ko_final)
(ko_final / "MANIFEST.md").write_text(
    f"""# tts/ko

- Source: {KO_REPO} (Meta MMS-TTS, VITS). PyTorch checkpoint exported to
  ONNX here (unlike Piper, which ships ONNX) -- see experiments/04_tts.ipynb
  section 5.
- License: MIT (MMS models) -- no espeak/GPL dependency, unlike the Piper
  stages.
- Packaged: {date.today().isoformat()}
- Precision: fp32. Exported with noise_scale=0 and noise_scale_duration=0
  for deterministic, reproducible output (VITS is stochastic by default);
  ONNX matches the PyTorch reference to <1e-2 on the probe sentences.
- Sample rate: {ko_sr}Hz.
- **Text front-end: uroman romanization is REQUIRED** (tokenizer is
  is_uroman=True; its raw vocab is only ~{ko_tok.vocab_size} Latin
  symbols). Python applies it automatically via the `uroman` package.
  The Rust port needs a Korean Hangul->Latin romanizer in front of the
  tokenizer -- this is the one TTS language that does NOT go through
  espeak; tracked as a real porting task in docs/TODO.md.
- Eval: round-trip TTS -> Whisper-small ASR, see section 5 below.

Files:
- `mms_tts_kor.onnx` (input_ids -> waveform, dynamic seq length)
- tokenizer files (`tokenizer_config.json`, `vocab.json`, ...) from
  save_pretrained -- the char->id map + is_uroman/normalize flags.
"""
)
print(f"  packaged -> models/tts/ko/  ({sorted(p.name for p in ko_final.iterdir())})")


In [ ]:
# --- Round-trip intelligibility (TTS -> ASR) for EN, ZH, KO ---
# Same idea as the VI check in section 3, but multilingual, so use the
# general-purpose Whisper-small (PyTorch here -- this is only a validation
# proxy, not a shipped artifact) which covers all three. Stacks TTS + ASR
# error, so it's a lower bound on TTS quality, not an exact measure.
from transformers import WhisperForConditionalGeneration, WhisperProcessor

wproc = WhisperProcessor.from_pretrained("openai/whisper-small")
wmodel = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").eval()

def roundtrip(path, lang):
    a, sr = sf.read(path)
    if a.ndim > 1:
        a = a.mean(axis=1)
    a16 = librosa.resample(a.astype(np.float32), orig_sr=sr, target_sr=16000) if sr != 16000 else a.astype(np.float32)
    feats = wproc(a16, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        gen = wmodel.generate(feats.input_features, language=lang, task="transcribe", max_new_tokens=64)
    return wproc.batch_decode(gen, skip_special_tokens=True)[0].strip()

all_synth = {"en": piper_synth["en"], "zh": piper_synth["zh"], "ko": ko_synth}
for lang in ("en", "zh", "ko"):
    print(f"=== {lang.upper()} ===")
    for sent, path in zip(tts_probe[lang], all_synth[lang]):
        heard = roundtrip(path, lang)
        print(f"  original: {sent}")
        print(f"  tts->asr: {heard}\n")

shutil.rmtree(piper_out)
